# 02_step1_Ecearth3historical_downscaling_F1 - Calibration of F1 (global)

**F1 = analogue-based downscaling** (your supervisor's method), learned between:

- **EC-Earth3 historical** at its native resolution (~0.7 deg), as input;
- **ERA5 0.25 deg**, as target.

No regridding, no WDF, no QDM: the analogue method does **not** require day-by-day consistency between EC-Earth3 and ERA5
(the two datasets are supplied separately, not paired in time).

**The method is not written here**: this notebook prepares the data and calls `train_f1` from `f1_downscaler.py`
(to be filled in with the supervisor's method).

F1 is then applied to all GCM runs (notebook step2), which gives the "ERA5-like" GCMs at 0.25 deg.

Whole globe (no region box). Volume: global ERA5 at 0.25 deg is ~1.5 GB per variable and per year (float32, uncompressed): the method will need to process the data in blocks (years, or spatial tiles).

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, warnings
from pathlib import Path
for _p in (Path.cwd(), Path.cwd().parent):   # ba_utils.py: same folder as the notebook, or the parent folder
    sys.path.insert(0, str(_p))
import numpy as np, xarray as xr, pandas as pd
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS = cfg["variables"]                      # ['pr', 'tasmax', 'tasmin']

TRAIN_PERIOD = ("1981", "2014")     # EC-Earth3 historical ends in 2014; for an out-of-sample test use ("1981", "2003")
import f1_downscaler as f1          # <- file to complete (supervisor's method)
F1_DIR = bu.out_dir(cfg, "plan_B", "F1_downscaling", f"{TRAIN_PERIOD[0]}-{TRAIN_PERIOD[1]}")
print("F1 folder:", F1_DIR)

## 1. Training data (not paired in time)

In [ ]:
gcm_hist = {v: bu.subset_time(bu.read_raw(cfg, "gcm_historical", v), *TRAIN_PERIOD) for v in VARS}
era5 = {v: bu.subset_time(bu.read_raw(cfg, "era5", v), *TRAIN_PERIOD) for v in VARS}
for v in VARS:
    g, e = gcm_hist[v], era5[v]
    print(f"{v:7s} native GCM: {g.sizes['lat']}x{g.sizes['lon']} (step ~{abs(float(np.diff(g.lat).mean())):.2f} deg, {g.sizes['time']} days)"
          f" | ERA5: {e.sizes['lat']}x{e.sizes['lon']} (step ~{abs(float(np.diff(e.lat).mean())):.2f} deg, {e.sizes['time']} days)")

## 2. Training (supervisor's method: to be written in `f1_downscaler.train_f1`)

In [ ]:
try:
    f1.train_f1(gcm_hist, era5, F1_DIR)
    print("F1 trained ->", F1_DIR)
except NotImplementedError as e:
    print("PLACEHOLDER:", e)

## 3. Notes

- Contract expected by notebook step2: `f1_downscaler.apply_f1(gcm_sim_dict, F1_DIR, target_lat, target_lon) -> dict` of 0.25 deg DataArrays (see `f1_downscaler.py`).
- Variables: `pr`, `tasmax`, `tasmin` (check `tasmin < tasmax` and `pr >= 0` after downscaling).
- Out-of-sample test (validations notebook): rerun with `TRAIN_PERIOD = ("1981", "2003")`, apply to EC-Earth3 historical 2004-2014 and compare the **distributions** with ERA5 2004-2014 (not day by day).